# Chapter 3 — Self-Sufficient Notebook: Tree-of-Thought Prompting Agent

This notebook converts the chapter's most important snippet into a runnable, self-sufficient Jupyter notebook.

## Why this snippet
I selected the **Tree-of-Thought (ToT) launch strategy example** because it is the clearest end-to-end demonstration of the chapter's central argument:

- prompts are not just instructions
- prompts can scaffold reasoning
- prompts can coordinate multiple expert perspectives
- prompt design becomes cognitive architecture

This notebook keeps that idea intact while making it runnable without an API key.

## What this notebook demonstrates

- a lightweight **PTCF** prompt blueprint
- a **mocked Tree-of-Thought workflow**
- three virtual experts:
  - market analyst
  - financial planner
  - marketing specialist
- a synthesis stage that combines their outputs
- a clear explanation of how the code reflects the chapter concepts

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any
import json
import textwrap

## Step 1 — Define a simple PTCF structure

The chapter emphasizes that prompt design should be structured around:

- **Persona**
- **Task**
- **Context**
- **Format**

We model that here explicitly so the reasoning flow stays readable.

In [ ]:
@dataclass
class PTCFPrompt:
    persona: str
    task: str
    context: str
    output_format: str

    def render(self) -> str:
        return (
            f"[PERSONA]\n{self.persona}\n\n"
            f"[TASK]\n{self.task}\n\n"
            f"[CONTEXT]\n{self.context}\n\n"
            f"[FORMAT]\n{self.output_format}"
        )

## Step 2 — Create a self-sufficient mock reasoning backend

Instead of calling a live LLM, we use a deterministic local generator.
This preserves the *architecture* of the chapter example while keeping the notebook runnable anywhere.

In [ ]:
class MockReasoner:
    def generate(self, prompt: str, variables: Dict[str, Any] | None = None) -> str:
        variables = variables or {}
        p = prompt.lower()

        if "virtual market analyst" in p:
            return (
                "Recommended target market: university students.\n"
                "Rationale: they have immediate need for structured learning tools, "
                "show stronger willingness to adopt AI study aids, and can be reached "
                "through focused digital channels more efficiently than high school "
                "students or enterprise buyers."
            )

        if "virtual financial planner" in p:
            market = variables.get("market", "")
            chosen_market = "university students" if "university" in market.lower() else "working professionals"
            return (
                f"Recommended revenue model: freemium subscription for {chosen_market}.\n"
                "Rationale: lowers adoption friction, supports recurring revenue, and "
                "creates a clean upgrade path for premium study plans and advanced analytics."
            )

        if "virtual marketing specialist" in p:
            market = variables.get("market", "")
            finance = variables.get("finance", "")
            return (
                "Recommended campaign:\n"
                "- Channels: student creators, campus ambassadors, SEO content, short-form video\n"
                "- Messaging: learn faster, personalize revision, reduce study overload\n"
                "- KPI: free-to-paid conversion rate within 30 days\n"
                f"- Grounding: market insight = {market[:80]}... | model = {finance[:80]}..."
            )

        if "strategic synthesis agent" in p:
            market = variables.get("market", "")
            finance = variables.get("finance", "")
            marketing = variables.get("marketing", "")
            return (
                "# Launch Strategy\n\n"
                "## Target Segment\n"
                "University students preparing for exams and structured coursework.\n\n"
                "## Revenue Model\n"
                "Freemium subscription with premium tiers for advanced planning, analytics, and AI study support.\n\n"
                "## Go-to-Market Plan\n"
                "Use creator-led awareness, campus ambassador partnerships, and content-led acquisition. "
                "Track conversion from free sign-up to paid plan as the primary KPI.\n\n"
                "## Notes\n"
                f"- Market analysis summary: {market[:120]}...\n"
                f"- Financial model summary: {finance[:120]}...\n"
                f"- Marketing summary: {marketing[:120]}..."
            )

        return "No reasoning rule matched this prompt."

reasoner = MockReasoner()

## Step 3 — Build the virtual expert prompts

This mirrors the chapter's ToT setup:
- one branch explores the market
- one branch explores the business model
- one branch explores the campaign
- a synthesis agent integrates the results

In [ ]:
market_prompt = PTCFPrompt(
    persona="You are a Virtual Market Analyst.",
    task=(
        "Evaluate the best target market for a new AI-powered educational app. "
        "Compare high school students, university students, and working professionals."
    ),
    context=(
        "You are helping a launch team choose the best initial market segment. "
        "Prioritize viability, urgency of need, and accessibility."
    ),
    output_format="Provide a concise recommendation with supporting rationale."
)

finance_prompt = PTCFPrompt(
    persona="You are a Virtual Financial Planner.",
    task=(
        "Recommend the optimal business model for the selected market: "
        "one-time purchase, subscription, or freemium."
    ),
    context=(
        "You must align the pricing model to the likely behavior and incentives "
        "of the selected segment."
    ),
    output_format="Provide a short recommendation and explain the economic logic."
)

marketing_prompt = PTCFPrompt(
    persona="You are a Virtual Marketing Specialist.",
    task=(
        "Design a focused awareness campaign for the selected target market and chosen revenue model."
    ),
    context=(
        "You should propose concrete channels, a message angle, and one measurable KPI."
    ),
    output_format="Return channels, messaging, and KPI in bullet form."
)

synthesis_prompt = PTCFPrompt(
    persona="You are a Strategic Synthesis Agent.",
    task=(
        "Integrate the expert analyses into a coherent product launch recommendation."
    ),
    context=(
        "You must reconcile the different expert viewpoints into one practical launch strategy."
    ),
    output_format=(
        "Output three sections: Target Segment, Revenue Model, and Go-to-Market Plan."
    )
)

## Step 4 — Run the Tree-of-Thought workflow

In [ ]:
market_result = reasoner.generate(market_prompt.render())
finance_result = reasoner.generate(
    finance_prompt.render(),
    variables={"market": market_result}
)
marketing_result = reasoner.generate(
    marketing_prompt.render(),
    variables={"market": market_result, "finance": finance_result}
)
final_strategy = reasoner.generate(
    synthesis_prompt.render(),
    variables={
        "market": market_result,
        "finance": finance_result,
        "marketing": marketing_result,
    }
)

print("MARKET ANALYST\n" + "-" * 60)
print(market_result)

print("\nFINANCIAL PLANNER\n" + "-" * 60)
print(finance_result)

print("\nMARKETING SPECIALIST\n" + "-" * 60)
print(marketing_result)

print("\nSYNTHESIZED STRATEGY\n" + "-" * 60)
print(final_strategy)

## What this snippet is all about

This snippet is a **self-contained simulation of Tree-of-Thought prompting**.

Instead of treating reasoning as one linear response, it creates several **virtual expert branches** that each look at the same product-launch problem from a different angle:

- the **market analyst** chooses the best target segment
- the **financial planner** chooses the most suitable business model
- the **marketing specialist** designs the launch campaign
- the **synthesis agent** merges those outputs into one strategy

This directly illustrates the concept discussed in the chapter: **advanced prompting can structure how an agent thinks**.  
The notebook also makes the **PTCF blueprint** concrete by showing how each expert prompt contains:
- a persona,
- a task,
- context,
- and an output format.

So the notebook is not just producing an answer; it is showing how prompt engineering becomes an **architectural method for reasoning, decomposition, and coordination**.

## Step 5 — Compare with a simple linear chain-of-thought style baseline

This helps clarify why the ToT pattern matters.

In [ ]:
def simple_chain_of_thought_launch_plan() -> str:
    return (
        "Step 1: Choose a target audience.\n"
        "Step 2: Choose a pricing model.\n"
        "Step 3: Pick marketing channels.\n"
        "Step 4: Produce a single launch recommendation.\n\n"
        "Recommendation: launch to university students using a freemium model "
        "and promote through creators and content marketing."
    )

cot_result = simple_chain_of_thought_launch_plan()
print(cot_result)

## Why the ToT version is stronger

The chain-of-thought baseline is useful, but it reasons through a **single path**.
The ToT version is closer to the chapter's core message because it separates perspectives and then synthesizes them.

That makes it a better fit for:
- strategic ambiguity
- competing constraints
- multi-perspective decision-making

## Suggested extensions

- replace `MockReasoner` with a real LLM backend
- add scoring for alternative branches
- add a critic agent to challenge weak assumptions
- serialize the expert outputs as JSON for downstream orchestration
- turn this into a true multi-agent workflow